# Week 12 Classwork Lab — From Bag of Words to Attention

**Format:** group classwork • **Duration:** 180 minutes •
**Lecture path:** Bag of Words → one-hot / IDs → embeddings → Word2Vec → RNN → attention → Q/K/V → scaled dot-product attention → self-/cross-/multi-head attention

## The challenge

You are not being asked to repeat lecture definitions. Your group must **make each representation fail, diagnose why it fails, and then justify the next representation**.

By the end, your group should be able to tell one continuous story:

> **What information is missing now, and what mechanism adds it next?**

### Group roles — rotate every mission

- **Driver:** edits/runs the notebook.
- **Skeptic:** challenges explanations and asks “What evidence do we have?”
- **Shape-checker:** checks vectors, matrices, dimensions, and row/column meanings.
- **Reporter:** writes the group's final explanation.


### Submission rule

For every mission, leave:
1. runnable code where code is requested;
2. a short written conclusion;
3. at least one piece of evidence: an output, a computed value, a diagram observation, or a paper section.

**Do not split the missions among people and paste answers together at the end.** The point is to reason as a group.

## 180-minute route

| Time | Mission | Main idea |
|---:|---|---|
| 5 min | Launch | Build the dependency map |
| 15 min | 1. The Collision Lab | What Bag of Words loses |
| 20 min | 2. Identity Is Not Meaning | One-hot, IDs, embeddings, similarity |
| 25 min | 3. Context Factory | Word2Vec / Skip-gram |
| 20 min | 4. The Recurrent Relay | RNN state, order, sequential bottleneck |
| 20 min | 5. Attention Rescue | Q/K/V and weighted retrieval |
| 25 min | 6. Matrix Control Room | Scaled dot-product attention, shapes, projections |
| 25 min | 7. Source Detective | Self-attention, paper + visual reading |
| 20 min | 8. Attention Architecture Cards | Cross-, multi-head, and local attention |
| 5 min | Exit ticket | Reconstruct the whole story |

> **Scope note:** The supplied lecture materials cover **RNNs** but do **not** introduce LSTM or GRU, so this classwork does not test LSTM/GRU.

## Launch — rebuild the lecture path without looking back (5 min)

As a group, put these cards in a defensible learning order:

`Word2Vec` • `Bag of Words` • `Q/K/V` • `one-hot` • `RNN` • `word ID` • `embedding` • `self-attention` • `scaled dot-product attention`

Then add **one short phrase above each arrow** explaining why you need to move to the next idea.

Example arrow phrase style: *“keeps identity, but still has no semantic geometry.”*

### Group answer

**Order:**  
`...`

**Arrow explanations:**  
- ...
- ...
- ...

# Mission 1 — The Collision Lab: make Bag of Words fail (15 min)

A Bag-of-Words representation counts vocabulary items. It remembers **which words and how many**, but not their sequence structure.

Your task is to behave like an adversary: find different meanings that collide under the representation.

In [ ]:
from collections import Counter
import re

def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())

def build_vocabulary(sentences):
    return sorted({w for s in sentences for w in tokenize(s)})

def bow_vector(sentence, vocabulary):
    counts = Counter(tokenize(sentence))
    return [counts[w] for w in vocabulary]

sentences = [
    "dog bites cat",
    "cat bites dog",
    "the chef praised the waiter",
    "the waiter praised the chef",
    "the movie was good",
    "the movie was not good",
]

vocab = build_vocabulary(sentences)
print("Vocabulary:", vocab)

for s in sentences:
    print(f"{s:32s} -> {bow_vector(s, vocab)}")

### 1A — Collision hunt

1. Find every pair above that gets the **same** vector.
2. For one collision, explain precisely what meaning-relevant information disappeared.
3. Invent a **new pair of sentences** that:
   - use exactly the same word counts,
   - mean noticeably different things,
   - and therefore collide under BoW.

Add your pair to the code cell below and verify the collision.

### 1B — The negation trap

`the movie was good` and `the movie was not good` do **not** have identical full BoW vectors because `not` is present in only one sentence.

So why is BoW still a weak representation for negation?

Your answer must distinguish:

- **detecting that the token `not` exists**, from
- **representing which word or phrase it changes**.

### 1C — Verdict

Complete this sentence in no more than 25 words:

> “Bag of Words is useful when __________, but it becomes dangerous when __________.”

In [ ]:
# Add your adversarial BoW collision here.
my_pair = [
    "REPLACE WITH SENTENCE A",
    "REPLACE WITH SENTENCE B",
]

my_vocab = build_vocabulary(my_pair)
for s in my_pair:
    print(s, "->", bow_vector(s, my_vocab))

# After editing, make this True:
# assert bow_vector(my_pair[0], my_vocab) == bow_vector(my_pair[1], my_vocab)

# Mission 2 — Identity Is Not Meaning (20 min)

The lecture moves through three ideas that are easy to confuse:

1. **word ID** — an address/index;
2. **one-hot vector** — explicit identity in a vocabulary-sized vector;
3. **embedding** — a dense learned vector with useful geometry.

Your group will prove that these are not interchangeable.

In [ ]:
import torch
import torch.nn.functional as F

words = ["cat", "dog", "car", "bus"]
word_to_id = {w: i for i, w in enumerate(words)}

def one_hot(word):
    v = torch.zeros(len(words))
    v[word_to_id[word]] = 1.0
    return v

for w in words:
    print(f"{w:>3s}  id={word_to_id[w]}  one-hot={one_hot(w).tolist()}")

### 2A — The ID renumbering trial

Imagine tomorrow we change the IDs to:

`cat → 91, dog → 4, car → 700, bus → 12`

without changing the model's embedding rows appropriately.

1. Does `91` make **cat** “more meaningful” than ID `4`?
2. Should semantic similarity be computed directly from raw IDs?
3. What does an embedding lookup table do that makes the ID itself unimportant?

Give a 3-sentence group answer.

In [ ]:
pairs = [("cat", "dog"), ("cat", "car"), ("dog", "bus")]

print("One-hot dot products:")
for a, b in pairs:
    print(a, b, "->", torch.dot(one_hot(a), one_hot(b)).item())

### 2B — The geometry test

For distinct one-hot vectors, the dot product is zero.

Explain why that causes the representation to treat:

- `cat` vs `dog`, and
- `cat` vs `bus`

as equally unrelated.

Then predict which pairs should be closest in the dense vectors below **before running the next cell**.

In [ ]:
toy_embeddings = torch.tensor([
    [0.90, 0.80, 0.10],  # cat
    [0.85, 0.75, 0.15],  # dog
    [0.10, 0.20, 0.90],  # car
    [0.15, 0.25, 0.85],  # bus
], dtype=torch.float)

normalized = F.normalize(toy_embeddings, dim=1)
similarity = normalized @ normalized.T

print("Cosine similarity matrix")
print(similarity.round(decimals=3))

### 2C — Representation lineup

For each representation, mark what it naturally gives you **without extra machinery**.

| Representation | Word identity? | Compact/dense? | Semantic geometry? | Depends on surrounding sentence? |
|---|---:|---:|---:|---:|
| raw word ID |  |  |  |  |
| one-hot |  |  |  |  |
| learned static embedding |  |  |  |  |

Then answer:

> If a learned embedding is better than one-hot, why do we still need something beyond a single static vector for the word **bank**?

# Mission 3 — Context Factory: build the training signal for Word2Vec (25 min)

The Skip-gram idea in the lecture is:

> use a **center word** to predict words found in a small **context window**.

Do not start by training. First make sure you understand exactly where the training examples come from.

In [ ]:
torch.manual_seed(7)

corpus = (
    "the cat sat on the mat the cat slept on the sofa "
    "the dog sat on the mat the dog slept on the sofa "
    "the car drove on the road the bus drove on the road"
).split()

vocab = sorted(set(corpus))
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}

print("Vocabulary:", vocab)
print("Corpus length:", len(corpus))

### 3A — Human pair generator

Without code, use this tiny sequence:

`the cat sat on mat`

with context window size `1`.

Write all `(center, context)` training pairs.

Now increase the window to `2`. What **new** pairs appear?

Only after your group agrees, complete the pair-builder below.

In [ ]:
def make_skipgram_pairs(tokens, word_to_id, window_size=2):
    pairs = []

    for center_index, center_word in enumerate(tokens):
        # TODO 1: compute the left boundary
        left = ...
        # TODO 2: compute the right boundary (remember Python's exclusive end)
        right = ...

        # TODO 3: loop over context positions and skip center_index
        # for context_index in ...:
        #     if ...:
        #         pairs.append((word_to_id[center_word], word_to_id[tokens[context_index]]))

    return pairs

# When finished:
# pairs = make_skipgram_pairs(corpus, stoi, window_size=2)
# print("number of pairs:", len(pairs))
# print("first 12:", [(itos[c], itos[t]) for c, t in pairs[:12]])

### 3B — Window-size experiment

Run the completed function with window sizes `1`, `2`, and `4`.

Record:

| Window | Number of training pairs | What kind of context becomes more common? |
|---:|---:|---|
| 1 |  |  |
| 2 |  |  |
| 4 |  |  |

Then debate:

> Is a larger window **always** better for learning meaning? Give one reason it might help and one reason it might blur useful local information.

### 3C — Finish the tiny Skip-gram model

The center word ID should first go through an **embedding table**.  
The resulting vector should then be scored against the **whole vocabulary**.

Complete only the two `TODO` lines in `__init__`.

In [ ]:
import torch.nn as nn

class TinySkipGram(nn.Module):
    def __init__(self, vocabulary_size, embedding_dim):
        super().__init__()

        # TODO: map a center-word ID to a dense vector.
        self.embedding = ...

        # TODO: map that dense vector to one score per vocabulary word.
        self.output = ...

    def forward(self, center_ids):
        z = self.embedding(center_ids)
        return self.output(z)

### 3D — Train, inspect, argue

After 3A and 3C are complete, run the cell below.

Do **not** grade the model by whether every neighbor is perfect; this is a deliberately tiny corpus. Grade your understanding by whether you can explain why the learned embedding rows move at all.

In [ ]:
# This cell assumes `pairs` exists from your completed 3A.
# It is provided to save class time.

if "pairs" not in globals():
    print("Complete Mission 3A first; `pairs` is not defined yet.")
else:
    centers = torch.tensor([c for c, _ in pairs])
    contexts = torch.tensor([t for _, t in pairs])

    model = TinySkipGram(len(vocab), embedding_dim=8)

    if not isinstance(getattr(model, "embedding", None), nn.Embedding):
        print("Complete the nn.Embedding TODO in TinySkipGram first.")
    elif not isinstance(getattr(model, "output", None), nn.Linear):
        print("Complete the nn.Linear TODO in TinySkipGram first.")
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=0.03)

        for epoch in range(250):
            optimizer.zero_grad()
            logits = model(centers)
            loss = F.cross_entropy(logits, contexts)
            loss.backward()
            optimizer.step()

        print("final training loss:", round(loss.item(), 4))

        with torch.no_grad():
            learned = F.normalize(model.embedding.weight, dim=1)

        def nearest_words(query, k=5):
            q = learned[stoi[query]]
            scores = learned @ q
            best = torch.topk(scores, k=min(k, len(vocab))).indices.tolist()
            return [(itos[i], round(scores[i].item(), 3)) for i in best]

        for query in ["cat", "dog", "car", "bus", "road", "sofa"]:
            print(query, "->", nearest_words(query))

### 3E — What Word2Vec solved, and what it did not

Answer all three:

1. Why can two words that occur in similar neighborhoods end up with similar vectors?
2. Why is the vector for a word such as `bank` still problematic if it is used in:
   - “the **bank** approved the loan”
   - “we sat beside the **bank** of the river”?
3. Complete the bridge:

> “Word2Vec makes word vectors meaningful using __________, but the vector is still __________ across different sentences.”

# Mission 4 — The Recurrent Relay (20 min)

A basic RNN updates a hidden state one token at a time:

\[
h_t = \tanh(x_t W_x + h_{t-1} W_h)
\]

Think of the hidden state as a message passed from one runner to the next.

In [ ]:
sequence_a = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
])

sequence_b = torch.flip(sequence_a, dims=[0])  # same vectors, reversed order

W_x = torch.tensor([[0.7, 0.2],
                    [0.1, 0.6]])
W_h = torch.tensor([[0.5, 0.0],
                    [0.0, 0.5]])

def run_tiny_rnn(sequence):
    h = torch.zeros(2)
    states = []

    for x_t in sequence:
        # TODO: one recurrent state update
        h = ...
        states.append(h.clone())

    return torch.stack(states)

# After filling the TODO:
# states_a = run_tiny_rnn(sequence_a)
# states_b = run_tiny_rnn(sequence_b)
# print("A states:\n", states_a)
# print("B states:\n", states_b)
# print("same final state?", torch.allclose(states_a[-1], states_b[-1]))

### 4A — BoW vs RNN: same pieces, different order

After completing the recurrence:

1. Do `sequence_a` and `sequence_b` end with the same hidden state?
2. Why can an RNN distinguish order when BoW cannot?
3. What information from token 1 reaches token 3 **directly**, and what must pass through intermediate hidden states?

### 4B — The dependency schedule

For a sequence of 8 tokens, draw the dependency chain:

`h0 → h1 → h2 → ... → h8`

Then answer:

- Can `h8` be computed before `h7`?
- Which part of this creates a parallel-training limitation?
- If important information begins at position 1 and matters at position 30, what is the risk of repeatedly compressing it through many hidden states?

### 4C — Design decision

Your group is given a sentence where the last word needs information from a word near the beginning.

Choose one:

- **RNN-only design**
- **a mechanism that can directly consult earlier positions**

Defend your choice in exactly **two sentences**. Your second sentence must identify a trade-off or limitation.

# Mission 5 — Attention Rescue: match, normalize, retrieve (20 min)

Use the lecture sentence:

> **The animal crossed the street because it was tired.**

Suppose the representation for **it** needs to retrieve information from useful positions such as **animal**.

Attention separates three jobs:

- **Query (Q):** what am I looking for?
- **Key (K):** how should I be matched?
- **Value (V):** what content should I contribute if selected?

In [ ]:
tokens = ["animal", "crossed", "street", "it", "tired"]

query = torch.tensor([1.0, 0.0])

keys = torch.tensor([
    [0.95, 0.05],  # animal
    [0.10, 0.90],  # crossed
    [0.05, 0.95],  # street
    [0.80, 0.20],  # it
    [0.20, 0.80],  # tired
])

values = torch.tensor([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
    [0.5, 0.5, 0.0],
    [0.2, 0.0, 0.8],
])

### 5A — Predict before computing

Without running a scoring cell:

1. Which token's Key should have the largest dot product with the Query?
2. Which token do you predict will get the largest softmax weight?
3. Does “largest weight” mean all other Values disappear? Explain.

In [ ]:
# Complete the three lines.

scores = ...                     # Query-Key matching
weights = ...                    # normalize across token positions
context = ...                    # weighted combination of Values

# Then uncomment:
# for token, score, weight in zip(tokens, scores, weights):
#     print(f"{token:8s} score={score.item():.3f} weight={weight.item():.3f}")
# print("weight sum:", weights.sum().item())
# print("context:", context)

### 5B — The Q/K/V role-swap incident

A teammate says:

> “Keys contain the information we return, and Values are what we compare with the Query.”

Diagnose the mistake.

Then use the database example below:

| Key | Value |
|---|---|
| Capital of Iran | Tehran |
| Capital of France | Paris |
| Capital of Japan | Tokyo |

For Query = “What is the capital of Iran?” explain:

1. what is used for **matching**;
2. what is **returned**;
3. why separating Key and Value can be useful.

### 5C — Hard choice vs soft mixture

Construct two outputs from the same attention weights:

- **hard retrieval:** keep only the Value with the largest weight;
- **soft retrieval:** use the weighted sum of all Values.

Which better matches the attention mechanism in the lecture? What information can soft retrieval preserve that hard retrieval discards?

# Mission 6 — Matrix Control Room (25 min)

Now every position has a Query, Key, and Value.

The lecture's matrix form is:

\[
\text{scores} = \frac{QK^T}{\sqrt{d_k}},
\qquad
\text{weights} = \text{softmax(scores)},
\qquad
\text{output} = \text{weights}V
\]

Your job is to make the implementation agree with the shapes and with the meaning of each matrix axis.

### 6A — Shape prediction: no code yet

Assume:

- `Q` has shape `[3, 4]`
- `K` has shape `[5, 4]`
- `V` has shape `[5, 6]`

Predict the shapes of:

1. `K.T`
2. `Q @ K.T`
3. attention weights
4. final output

Also explain why the number `5` appears in both `K` and `V`.

In [ ]:
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)

    # TODO 1: Query-Key scores with the lecture scaling
    scores = ...

    if mask is not None:
        # TODO 2: forbidden score locations must receive negative infinity
        scores = ...

    # TODO 3: normalize over key positions
    weights = ...

    # TODO 4: retrieve/mix the Values
    output = ...

    return output, weights, scores

In [ ]:
Q_demo = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
])

K_demo = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
])

V_demo = torch.tensor([
    [10.0, 0.0],
    [0.0, 20.0],
    [5.0, 5.0],
])

# After completing the function:
# out, W, S = scaled_dot_product_attention(Q_demo, K_demo, V_demo)
# print("scores:\n", S)
# print("weights:\n", W)
# print("row sums:", W.sum(dim=-1))
# print("output:\n", out)

### 6B — Read one row like a sentence

Pick one row `i` of your attention matrix.

Write:

- “The Query belongs to position ___.”
- “Its strongest Key match is position ___.”
- “Therefore its output receives the largest contribution from Value ___.”
- “The row sums to 1 because ___.”

Now inspect two opposite entries `W[i, j]` and `W[j, i]`.

> Must they be equal? Why or why not?

Use your actual matrix as evidence.

### 6C — Why divide by \(\sqrt{d_k}\)? An experiment

You will compare softmax behavior with and without scaling.

The supplied code creates larger dot products by repeating dimensions. Complete the two score formulas, then compare the maximum attention probability.

In [ ]:
torch.manual_seed(3)

Q_big = torch.randn(4, 64)
K_big = torch.randn(4, 64)

# TODO:
unscaled_scores = ...
scaled_scores = ...

# TODO:
unscaled_weights = ...
scaled_weights = ...

# Then uncomment:
# print("mean row max WITHOUT scaling:", unscaled_weights.max(dim=-1).values.mean().item())
# print("mean row max WITH scaling   :", scaled_weights.max(dim=-1).values.mean().item())

Explain what you observe.

Your explanation must use the ideas **dot-product magnitude**, **softmax**, and **peaked distribution**.  
Do not simply write “scaling makes training stable.”

### 6D — Where do Q, K, and V come from?

Before running, predict every shape.

\[
Q=XW_Q,\quad K=XW_K,\quad V=XW_V
\]

In [ ]:
torch.manual_seed(42)

X = torch.randn(5, 4)
W_Q = torch.randn(4, 3)
W_K = torch.randn(4, 3)
W_V = torch.randn(4, 3)

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print("X:", X.shape)
print("Q:", Q.shape, "K:", K.shape, "V:", V.shape)

# Once your attention function is complete:
# projected_out, projected_weights, _ = scaled_dot_product_attention(Q, K, V)
# print("attention matrix:", projected_weights.shape)
# print("output:", projected_out.shape)

Explain why using **three learned projections of the same token representation** is more flexible than forcing one vector to serve the roles of Query, Key, and Value identically.

# Mission 7 — Source Detective: self-attention and attention sources (25 min)

This mission has two external sources. Do not read everything. Use the specified sections to answer precise questions.

## Source A — *Attention Is All You Need*

Paper: <https://arxiv.org/abs/1706.03762>

Read only:

- **§3.2.1 — Scaled Dot-Product Attention**
- **§3.2.2 — Multi-Head Attention**
- **§3.2.3 — Applications of Attention in our Model**

For the **self-attention questions**, your evidence must come specifically from **§3.2.3**.

### 7A — Paper scavenger hunt

Answer with a short quotation fragment **or** a faithful paraphrase plus the section number.

1. **Scaled dot product (§3.2.1):**  
   What is the attention formula? What quantity is used to scale the dot products?

2. **Why scale? (§3.2.1):**  
   What problem can large \(d_k\) create for dot products and softmax?

3. **Encoder self-attention (§3.2.3):**  
   Where do the encoder's Q, K, and V come from?

4. **Encoder–decoder attention (§3.2.3):**  
   Where do the Queries come from, and where do Keys/Values come from?

5. **Multi-head attention (§3.2.2):**  
   Why use several projected attention heads instead of one single full-dimensional attention operation?

### Evidence table

| Question | Your answer | Paper section / figure |
|---|---|---|
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |
| 4 |  |  |
| 5 |  |  |


## Source B — Jay Alammar, *The Illustrated Transformer*

Open: <https://jalammar.github.io/illustrated-transformer/>

Focus on these parts:

- **Self-Attention at a High Level**
- **Self-Attention in Detail**
- **Matrix Calculation of Self-Attention**
- **The Beast With Many Heads**

### 7B — Diagram-to-code translation

Find the visual explanation where the word **“it”** can associate with **“animal.”**

Discuss as a group:

1. How is that visual story similar to the lecture's “animal … it … tired” attention example?
2. In **Self-Attention in Detail**, map the illustrated steps to your code:
   - create Q/K/V;
   - Query–Key dot products;
   - divide by \(\sqrt{d_k}\);
   - softmax;
   - weight Values;
   - sum Values.
3. Write one sentence describing something the visualization makes clearer than the formula alone.
4. Write one sentence describing something the visualization may make look simpler than it really is.

Your reporter should be ready to explain the full six-step calculation without reading from the page.

# Mission 8 — Attention Architecture Cards (20 min)

You now know the core computation. The remaining question is:

> **Who supplies Q/K/V, and which positions are visible?**

Use that question to separate self-attention, cross-attention, multi-head attention, and one historical form of local attention.

## 8A — Cross-attention shape puzzle

Given:

- decoder Queries: `[2, 4]`
- encoder Keys: `[5, 4]`
- encoder Values: `[5, 6]`

Predict:

1. attention-weight shape;
2. output shape;
3. what each row of the weight matrix corresponds to;
4. what each column corresponds to.

Then verify with code.

In [ ]:
decoder_queries = torch.randn(2, 4)
encoder_keys = torch.randn(5, 4)
encoder_values = torch.randn(5, 6)

# After your attention function works:
# cross_out, cross_weights, _ = scaled_dot_product_attention(
#     decoder_queries, encoder_keys, encoder_values
# )
# print("weights:", cross_weights.shape)
# print("output :", cross_out.shape)

## 8B — Multi-head attention: specialist committee

A teammate claims:

> “Head 1 is always syntax, Head 2 is always coreference, and Head 3 is always negation.”

Use the lecture and *Attention Is All You Need* §3.2.2 to critique this claim.

Then explain why multiple heads can be useful **without assigning a fixed human label to each head**.

Your answer must include the phrase:

> “different learned projection subspaces”

## 8C — Local attention: a short historical extension

Paper: Minh-Thang Luong, Hieu Pham, Christopher D. Manning,  
**“Effective Approaches to Attention-based Neural Machine Translation” (2015)**  
<https://arxiv.org/abs/1508.04025>

Read only:

- **§3.1 — Global Attention**
- **§3.2 — Local Attention**
- look at the paper's **global vs local attention figures**.

This paper uses attention inside an RNN encoder–decoder setting, so treat it as a **historical attention design**, not as a claim that it is identical to every modern “local Transformer attention” method.

### 8C-1 — Global or local?

Answer from §3.1–§3.2:

1. What does **global attention** consider when creating its context?
2. What does **local attention** restrict?
3. What computational or modeling motivation do the authors give for looking at only a subset of source positions?

### 8C-2 — Window exercise

Suppose a simplified local mechanism attends only inside:

\[
[p_t-D,\; p_t+D]
\]

For source positions `1 ... 12`, let:

- \(p_t = 7\)
- \(D = 2\)

Which source positions are visible?

Now compare with full attention over all 12 positions:

- What work is avoided?
- What important dependency could be missed?

### 8C-3 — local-m vs local-p

From **§3.2**, identify the key conceptual difference between the paper's:

- **local-m (monotonic)** approach;
- **local-p (predictive)** approach.

Do not copy equations unless they help your explanation. Explain the difference in your own words.

## 8D — Architecture card sort

For each card, identify the most appropriate mechanism and justify it using **Q source, K/V source, and visibility**.

### Card A
Every word in an encoder can use every word in the same input sequence.

### Card B
Two decoder positions retrieve information from five encoder positions.

### Card C
Each target step examines only a small source neighborhood around a chosen center.

Choose from:

- encoder self-attention
- cross-attention
- local attention

### Challenge card E

A mechanism runs several attention calculations in parallel using different learned Q/K/V projections, then combines the results.

What mechanism is this? Why is it not defined mainly by a visibility mask?

# Exit ticket — reconstruct the whole story (5 min)

No notes for the first 3 minutes.

### A. The chain

Fill every blank with a **limitation or improvement**, not just a noun.

> Bag of Words → __________ → one-hot / IDs → __________ → embeddings → __________ → Word2Vec → __________ → RNN → __________ → attention

### B. Three fast diagnoses

1. A student says:  
   **“Word ID 400 is more similar to word ID 401 than to word ID 9.”**  
   What is wrong?

2. A student says:  
   **“The largest attention weight tells us which Key vector is returned.”**  
   What is wrong?

3. A student says:  
   **“Self-attention and cross-attention use different math, so their output shapes follow different rules.”**  
   What is wrong or incomplete?

### C. One matrix sentence

Complete accurately:

> “Row \(i\), column \(j\) of an attention-weight matrix tells us __________.”

### D. One-minute group report

Your reporter must explain, in order:

**BoW → embedding → Word2Vec → RNN → Q/K/V attention → self-attention / cross-attention**

in **six sentences maximum**.

# Completion checklist

Before submitting, make sure your group has:

- [ ] produced and explained a Bag-of-Words collision;
- [ ] distinguished word IDs, one-hot vectors, and dense embeddings;
- [ ] generated Skip-gram center/context pairs;
- [ ] completed and trained the tiny Word2Vec-style model;
- [ ] shown how reversing a sequence affects an RNN;
- [ ] computed one-query attention using Q/K/V;
- [ ] completed scaled dot-product attention;
- [ ] interpreted at least one attention-matrix row;
- [ ] explained the \(\sqrt{d_k}\) scaling experiment;
- [ ] predicted shapes for learned Q/K/V projections;
- [ ] used **Attention Is All You Need §3.2.1–§3.2.3** as evidence;
- [ ] used **The Illustrated Transformer** to map visuals to computation;
- [ ] distinguished self-attention from cross-attention;
- [ ] explained multi-head attention without assigning fixed human meanings to heads;
- [ ] read **Luong et al. (2015) §3.1–§3.2** and compared global vs local attention.

## Optional fast-finisher challenge

Create a 4×4 attention-weight matrix of your own that satisfies all of the following:

1. every row sums to 1;
2. it is directional (not symmetric);
3. at least one row spreads attention across three different positions;
4. row 4 attends most strongly to position 1.

Then explain a hypothetical language relationship that your row 4 could represent.